In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# Config
# ============================================================
BASE_DIR = Path("./")

INPUT_GFEVD = BASE_DIR / "gfevd_all.npy"
INPUT_DATE = BASE_DIR / "tvpvar_input_scaled.csv"
INPUT_EVENTS = BASE_DIR / "events_window0.csv"
INPUT_PAIRWISE = BASE_DIR / "event_quantification_window0_pairwise_long.csv"

OUT_DIR = BASE_DIR / "solvpn_copper_pair_outputs"
OUT_DIR.mkdir(exist_ok=True)

VAR_NAMES = [
    "SOLVPN",
    "COPPER",
    "GOLD",
    "SILVER",
    "DXY",
    "UST10Y",
    "VIX"
]

PAIR_A_FROM = "SOLVPN"
PAIR_A_TO = "COPPER"

PAIR_B_FROM = "COPPER"
PAIR_B_TO = "SOLVPN"

H = 20                  # persistence horizon
WEAK_RATIO = 0.30       # bilateral / irrelevant 판정용
USE_ABS_AUC = True

# ============================================================
# 1. Load
# ============================================================
gfevd = np.load(INPUT_GFEVD)   # shape: (T, k, k)

df_date = pd.read_csv(INPUT_DATE)
df_date["Date"] = pd.to_datetime(df_date["Date"])

df_events = pd.read_csv(INPUT_EVENTS)
df_events["Date"] = pd.to_datetime(df_events["Date"])

df_pair = pd.read_csv(INPUT_PAIRWISE)
df_pair["Date"] = pd.to_datetime(df_pair["Date"])

T, k, _ = gfevd.shape

if len(VAR_NAMES) != k:
    raise ValueError(f"VAR_NAMES length ({len(VAR_NAMES)}) != gfevd dimension ({k})")

dates = pd.to_datetime(df_date["Date"].values[-T:])

print(f"[INFO] T={T}, k={k}")
print(f"[INFO] Number of events: {len(df_events)}")

# ============================================================
# 2. Build pairwise level series S_ij(t)
# ============================================================
var_to_idx = {v: i for i, v in enumerate(VAR_NAMES)}

a_i = var_to_idx[PAIR_A_FROM]
a_j = var_to_idx[PAIR_A_TO]

b_i = var_to_idx[PAIR_B_FROM]
b_j = var_to_idx[PAIR_B_TO]

S_sc = gfevd[:, a_i, a_j]   # SOLVPN -> COPPER
S_cs = gfevd[:, b_i, b_j]   # COPPER -> SOLVPN

df_level = pd.DataFrame({
    "Date": dates,
    "S_SOLVPN_to_COPPER": S_sc,
    "S_COPPER_to_SOLVPN": S_cs
})

date_to_idx_level = {pd.Timestamp(d): i for i, d in enumerate(dates)}

# ============================================================
# 3. Pairwise delta at event date from existing pairwise long file
# ============================================================
# event_quantification_window0_pairwise_long.csv:
# columns: Date, From, To, DeltaS
pair_sc = df_pair[
    (df_pair["From"] == PAIR_A_FROM) & (df_pair["To"] == PAIR_A_TO)
][["Date", "DeltaS"]].rename(columns={"DeltaS": "DeltaS_SOLVPN_to_COPPER"})

pair_cs = df_pair[
    (df_pair["From"] == PAIR_B_FROM) & (df_pair["To"] == PAIR_B_TO)
][["Date", "DeltaS"]].rename(columns={"DeltaS": "DeltaS_COPPER_to_SOLVPN"})

df_pair_event = pd.merge(pair_sc, pair_cs, on="Date", how="outer").sort_values("Date").reset_index(drop=True)

# event 날짜만 유지
df_pair_event = pd.merge(
    df_events[["Date"]],
    df_pair_event,
    on="Date",
    how="left"
).sort_values("Date").reset_index(drop=True)

# ============================================================
# 4. Persistence function for pair-specific path
# ============================================================
def compute_pair_persistence(level_series, event_idx, horizon, use_abs_auc=True):
    """
    level_series: S_ij(t) level series
    event_idx   : index in level series
    path_h      : S(t_e + h) - S(t_e - 1)
    """
    if event_idx == 0:
        return None

    max_h = min(horizon, len(level_series) - 1 - event_idx)
    if max_h < 0:
        return None

    baseline = level_series[event_idx - 1]
    path = np.array([
        level_series[event_idx + h] - baseline
        for h in range(max_h + 1)
    ], dtype=float)

    abs_path = np.abs(path)

    peak = float(abs_path.max())
    peak_h = int(abs_path.argmax())

    initial = float(abs_path[0])

    if initial == 0:
        half_life = 0
        duration_above_half = 0
    else:
        half_threshold = 0.5 * initial
        half_candidates = np.where(abs_path <= half_threshold)[0]
        half_life = int(half_candidates[0]) if len(half_candidates) > 0 else np.nan
        duration_above_half = int(np.sum(abs_path >= half_threshold))

    auc = float(abs_path.sum()) if use_abs_auc else float(path.sum())

    return {
        "Baseline": float(baseline),
        "InitialResponse": float(path[0]),
        "Peak": peak,
        "PeakHorizon": peak_h,
        "HalfLife": half_life,
        "DurationAboveHalf": duration_above_half,
        "AUC": auc,
        "MaxHorizonUsed": int(max_h),
        "Path": path
    }

# ============================================================
# 5. Event-wise pair summary
# ============================================================
summary_rows = []
path_rows = []

for _, row in df_events.iterrows():
    event_date = pd.Timestamp(row["Date"])

    if event_date not in date_to_idx_level:
        print(f"[WARN] Event {event_date.date()} not found in aligned dates, skipped")
        continue

    event_idx = date_to_idx_level[event_date]

    res_sc = compute_pair_persistence(S_sc, event_idx, H, use_abs_auc=USE_ABS_AUC)
    res_cs = compute_pair_persistence(S_cs, event_idx, H, use_abs_auc=USE_ABS_AUC)

    if (res_sc is None) or (res_cs is None):
        print(f"[WARN] Event {event_date.date()} persistence not computable, skipped")
        continue

    # event-date delta from existing pairwise file
    match = df_pair_event[df_pair_event["Date"] == event_date]
    if len(match) == 0:
        delta_sc = np.nan
        delta_cs = np.nan
    else:
        delta_sc = float(match["DeltaS_SOLVPN_to_COPPER"].iloc[0]) if pd.notna(match["DeltaS_SOLVPN_to_COPPER"].iloc[0]) else np.nan
        delta_cs = float(match["DeltaS_COPPER_to_SOLVPN"].iloc[0]) if pd.notna(match["DeltaS_COPPER_to_SOLVPN"].iloc[0]) else np.nan

    # asymmetry
    if pd.isna(delta_sc) or pd.isna(delta_cs):
        asym = np.nan
        abs_asym = np.nan
        pair_total_delta = np.nan
        pair_share_sc = np.nan
        pair_share_cs = np.nan
        relationship_type = "unknown"
    else:
        asym = delta_sc - delta_cs
        abs_asym = abs(asym)
        pair_total_delta = delta_sc + delta_cs

        if pair_total_delta == 0:
            pair_share_sc = 0.0
            pair_share_cs = 0.0
            relationship_type = "weak_or_irrelevant"
        else:
            pair_share_sc = delta_sc / pair_total_delta
            pair_share_cs = delta_cs / pair_total_delta

            # 유형 분류
            # 둘 다 충분히 크면 bilateral
            if (pair_share_sc >= WEAK_RATIO) and (pair_share_cs >= WEAK_RATIO):
                relationship_type = "bilateral"
            else:
                relationship_type = "SOLVPN_driven" if delta_sc > delta_cs else "COPPER_driven"

            # 둘 다 아주 작으면 weak_or_irrelevant
            # 전체 event에서 관심 pair 비중이 낮은 경우를 잡기 위해 추가 규칙
            if pair_total_delta < 1e-12:
                relationship_type = "weak_or_irrelevant"

    summary_rows.append({
        "Date": event_date,

        "DeltaS_SOLVPN_to_COPPER": delta_sc,
        "DeltaS_COPPER_to_SOLVPN": delta_cs,
        "PairDeltaTotal": pair_total_delta,
        "Asymmetry_DeltaS": asym,
        "AbsAsymmetry_DeltaS": abs_asym,
        "Share_SOLVPN_to_COPPER": pair_share_sc,
        "Share_COPPER_to_SOLVPN": pair_share_cs,

        "SC_Baseline": res_sc["Baseline"],
        "SC_InitialResponse": res_sc["InitialResponse"],
        "SC_Peak": res_sc["Peak"],
        "SC_PeakHorizon": res_sc["PeakHorizon"],
        "SC_HalfLife": res_sc["HalfLife"],
        "SC_DurationAboveHalf": res_sc["DurationAboveHalf"],
        "SC_AUC": res_sc["AUC"],
        "SC_MaxHorizonUsed": res_sc["MaxHorizonUsed"],

        "CS_Baseline": res_cs["Baseline"],
        "CS_InitialResponse": res_cs["InitialResponse"],
        "CS_Peak": res_cs["Peak"],
        "CS_PeakHorizon": res_cs["PeakHorizon"],
        "CS_HalfLife": res_cs["HalfLife"],
        "CS_DurationAboveHalf": res_cs["DurationAboveHalf"],
        "CS_AUC": res_cs["AUC"],
        "CS_MaxHorizonUsed": res_cs["MaxHorizonUsed"],

        "RelationshipType": relationship_type
    })

    for h, val in enumerate(res_sc["Path"]):
        path_rows.append({
            "Date": event_date,
            "Direction": "SOLVPN_to_COPPER",
            "Horizon": h,
            "Response": float(val)
        })

    for h, val in enumerate(res_cs["Path"]):
        path_rows.append({
            "Date": event_date,
            "Direction": "COPPER_to_SOLVPN",
            "Horizon": h,
            "Response": float(val)
        })

# ============================================================
# 6. Save
# ============================================================
df_summary = pd.DataFrame(summary_rows).sort_values("Date").reset_index(drop=True)
df_path = pd.DataFrame(path_rows).sort_values(["Date", "Direction", "Horizon"]).reset_index(drop=True)

summary_path = OUT_DIR / "solvpn_copper_event_pair_summary.csv"
path_path = OUT_DIR / "solvpn_copper_event_pair_path_long.csv"

df_summary.to_csv(summary_path, index=False)
df_path.to_csv(path_path, index=False)

print(f"[INFO] Saved summary: {summary_path}")
print(f"[INFO] Saved path long: {path_path}")
print("[INFO] Done")

[INFO] T=1035, k=7
[INFO] Number of events: 52
[INFO] Saved summary: solvpn_copper_pair_outputs\solvpn_copper_event_pair_summary.csv
[INFO] Saved path long: solvpn_copper_pair_outputs\solvpn_copper_event_pair_path_long.csv
[INFO] Done
